In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del folder production_company

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
production_companies_df = (
    spark.read
        .format("csv")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/production_company")
) 
display(production_companies_df)

In [0]:
production_companies_df_shema = StructType([
    StructField("companyId", IntegerType(), True),
    StructField("companyName", StringType(), True)
])
production_companies_df = (
    spark.read
        .format('csv')
        .schema(production_companies_df_shema)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/production_company")
)
display(production_companies_df)
production_companies_df.printSchema()

In [0]:
production_companies_df = (
    add_ingestion_date(production_companies_df)
    .select(
        col("companyId").alias("company_id"),
        col("companyName").alias("company_name"),
        lit(v_env).alias("env"),
        lit(v_file_date).alias("file_date")
    )
)
display(production_companies_df)

In [0]:
overwrite_partitioned_table("movie_silver", "production_companies", "file_date", v_file_date)

In [0]:
(
    production_companies_df.write
    .mode("append")
    .partitionBy("file_date")
    .saveAsTable("movie_silver.production_companies")
)

In [0]:
dbutils.notebook.exit("Success")